Project: Frontend UI library converter

Business Justification: moving from one frontend UI library to another can be an hazzle. While all other aspects of a project can be built on abstractions, the ui framewoor itself cannot be abstracted. This project makes it easy to convert components (React or Vue components) from one ui library (eg Veutify) to another UI library (eg Nuxt UI built on Tailwind).

First the imports

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import gradio as gr
import subprocess
from pathlib import Path
import socket
from pathlib import Path

load API Keys

In [2]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"


Lets setup the model

In [3]:
def chat(message,model,sourceFramework,destinationFramework):
    print(message,model,sourceFramework,destinationFramework)
    client = OpenAI() if model == "OpenAI" else OpenAI(base_url= "https://generativelanguage.googleapis.com/v1beta/openai/",api_key=GEMINI_API_KEY)
    system_prompt = "your task is to convert the "+sourceFramework+" markup to "+destinationFramework+" markup. Always return a fully functional component that can be previewed."
    messages = [ {"role":"system","content":system_prompt}, {"role":"user","content":message} ]
    response = client.chat.completions.create(
        model= "gpt-5" if model == "OpenAI" else "gemini-2.0-flash",
        messages=messages
    )
    message = response.choices[0].message.content

    nuxt_preview_app = Path.cwd() / "preview_apps/nuxt/preview-app"
    install_nuxt_dependencies(nuxt_preview_app)
    run_nuxt_dev_server(nuxt_preview_app)
    write_preview_component(nuxt_preview_app,message)
    

    return message
    

Now lets setup our preview apps and Gradio

In [ ]:

with gr.Blocks() as ui:
    with gr.Row(equal_height=True):
        with gr.Column():
            code_input = gr.Code(interactive=True,language="html",lines=20,max_lines=20)
            with gr.Row():
                model_picker = gr.Dropdown(show_label=False,choices=["OpenAI","Gemini"])
                source_framework_picker = gr.Dropdown(show_label=False,choices=["Veutify","Nuxt UI"])
        with gr.Column():
            code_output = gr.Code(interactive=False,language="html",lines=20,max_lines=20)
            with gr.Row():
                destination_framework_picker = gr.Dropdown(show_label=False,choices=["Veutify","Nuxt UI"])
                convert_button = gr.Button("Convert",interactive=True,)
    convert_button.click(fn=chat,inputs=[code_input,model_picker,source_framework_picker,destination_framework_picker],outputs=[code_output])
    preview = gr.HTML(
        """
        <iframe
            src="http://localhost:3000"
            style="width:100%; height:600px; border:none;"
        ></iframe>
        """
    ,max_height=400)
ui.launch()
                


setup preview projects

In [5]:
def install_nuxt_dependencies(sandbox_path: Path):
    """
    Install npm dependencies only if node_modules does not exist.
    """
    node_modules = sandbox_path / "node_modules"
    if node_modules.exists():
        print("🟢 Dependencies already installed.")
        return

    print("📦 Installing Nuxt dependencies...")
    subprocess.run(
        ["npm", "install"],
        cwd=sandbox_path,
        check=True
    )
    print("✅ Dependencies installed successfully.")

run the nuxt project

In [6]:
def run_nuxt_dev_server(sandbox_path: Path, port: int = 3000):
    """
    Start Nuxt dev server if it's not already running.
    """
    if is_port_in_use(port):
        print(f"🟢 Nuxt dev server already running on port {port}")
        return

    print(f"🚀 Starting Nuxt dev server on port {port}...")
    # Popen runs it in the background
    subprocess.Popen(
        ["npm", "run", "dev"],
        cwd=sandbox_path
    )
    print("✅ Nuxt dev server started.")

In [7]:
def write_preview_component(sandbox_path: Path, nuxt_code: str):
    """
    Overwrites Preview.vue with the generated component code.
    """
    preview_file = sandbox_path / "app/components" / "preview.vue"
    preview_file.parent.mkdir(parents=True, exist_ok=True)  # ensure folder exists

    with open(preview_file, "w", encoding="utf-8") as f:
        f.write(nuxt_code)

    print(f"🖌 Preview.vue updated at {preview_file}")

In [8]:
def is_port_in_use(port: int = 3000) -> bool:
    """Return True if a TCP port is in use."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0